## Example 1: Basic pamela authentication interface

In [ ]:
import inspect

import pamela

# Example 1: the authentication entry point and its error type
print("Example 1: pamela authentication interface")

sig = inspect.signature(pamela.authenticate)
print(f"authenticate signature: {sig}")
assert list(sig.parameters)[:3] == ['username', 'password', 'service']
assert sig.parameters['service'].default == 'login'

# PAMError is an ordinary exception, and PAM_SUCCESS is PAM's zero return code
assert issubclass(pamela.PAMError, Exception)
assert pamela.PAM_SUCCESS == 0

pam_constants = sorted(attr for attr in dir(pamela) if attr.startswith('PAM_'))
print(f"PAM constants exported: {len(pam_constants)}")
print(f"Sample constants: {pam_constants[:5]}")

print("Example 1 finished")

## Example 2: PAM handle items

In [ ]:
import pamela

# Example 2: open a PAM handle (no authentication happens) and read/write its items
print("Example 2: PAM handle items")

for name in ('open_session', 'close_session', 'check_account', 'change_password'):
    assert callable(getattr(pamela, name)), f"pamela.{name} is missing"

handle = pamela.pam_start('login', 'no-such-user-xyz')
try:
    assert isinstance(handle, pamela.PamHandle)
    service = handle.get_item(pamela.PAM_SERVICE)
    user = handle.get_item(pamela.PAM_USER)
    print(f"PAM_SERVICE={service!r} PAM_USER={user!r}")
    assert service == 'login'
    assert user == 'no-such-user-xyz'

    handle.set_item(pamela.PAM_TTY, 'pts/9')
    tty = handle.get_item(pamela.PAM_TTY)
    print(f"PAM_TTY after set_item: {tty!r}")
    assert tty == 'pts/9'
finally:
    pamela.pam_end(handle)

print("Example 2 finished")

## Example 3: PAM error handling and constants

In [ ]:
import pamela

# Example 3: PAMError formatting and the standard PAM constants
print("Example 3: PAM error handling and constants")

test_error = pamela.PAMError("Test error message", errno=1)
print(f"Test error created: {test_error}")
assert test_error.message == "Test error message"
assert test_error.errno == 1
assert str(test_error) == "[PAM Error 1] Test error message"

# Without a message, PAMError looks the text up with pam_strerror()
looked_up = pamela.PAMError(errno=7)
print(f"pam_strerror(7): {looked_up.message!r}")
assert looked_up.errno == 7
assert isinstance(looked_up.message, str) and looked_up.message

# PAM item identifiers must be distinct
item_constants = {name: getattr(pamela, name)
                  for name in ('PAM_SERVICE', 'PAM_USER', 'PAM_TTY', 'PAM_RUSER', 'PAM_RHOST')}
print(f"PAM item constants: {item_constants}")
assert len(set(item_constants.values())) == len(item_constants)

print("Example 3 finished")

## Example 4: PAM environment and conversation callbacks

In [ ]:
import pamela

# Example 4: conversation callbacks and the PAM environment of a handle
print("Example 4: PAM environment and conversation callbacks")

conv = pamela.new_simple_password_conv(['secret'], 'utf-8')
assert callable(conv)

msg_types = {name: getattr(pamela, name)
             for name in ('PAM_PROMPT_ECHO_OFF', 'PAM_PROMPT_ECHO_ON', 'PAM_ERROR_MSG', 'PAM_TEXT_INFO')}
print(f"PAM message styles: {msg_types}")
assert len(set(msg_types.values())) == len(msg_types)

handle = pamela.pam_start('login', 'no-such-user-xyz')
try:
    handle.put_env('SMOKE_TEST', 'yes')
    assert handle.get_env('SMOKE_TEST') == 'yes'
    assert handle.get_envlist().get('SMOKE_TEST') == 'yes'
    print(f"PAM environment: {handle.get_envlist()}")

    handle.del_env('SMOKE_TEST')
    try:
        handle.get_env('SMOKE_TEST')
    except pamela.PAMError:
        print("get_env() raises PAMError for a deleted variable")
    else:
        raise AssertionError("get_env() returned a deleted variable")
finally:
    pamela.pam_end(handle)

print("Example 4 finished")

## Example 5: Rejected authentication attempt

In [ ]:
import pamela

# Example 5: a real authentication attempt that must fail; it needs no root access
print("Example 5: rejected authentication attempt")

try:
    pamela.authenticate('no-such-user-xyz', 'wrong-password', service='login')
except pamela.PAMError as e:
    print(f"authenticate() rejected the unknown user: {e}")
    assert e.errno is not None and e.errno != pamela.PAM_SUCCESS
else:
    raise AssertionError("authenticate() accepted an unknown user")

print("Example 5 finished")